In [1]:
# %pip uninstall numpy scipy scikit-learn -y

# # 使用 conda 安裝
# %conda install numpy=1.24.3 scipy=1.10.1 scikit-learn=1.3.0 -c conda-forge

In [1]:
import os
import mne


import numpy as np
import pickle
import random
from sklearn.model_selection import train_test_split
from scipy.signal import detrend

In [2]:
rs=42 #set seed

channel_size = 30
output_path = f'augmented_data/Stress_noleak_{channel_size}chan_no400up_swien42'

# Process files for each split
sfreq = 200
increase_step_size = 250  # 3.75 seconds overlap for increase class
normal_step_size = 1000   # No overlap for normal class

print(f"Set random seed as: {rs}")
print(f"Set step size of increase file as: {increase_step_size} ({(normal_step_size-increase_step_size)/sfreq}seconds overlap)")
print(f"Set step size of normal file as: {normal_step_size} (No overlap for normal class)")

subset_channels = ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'FT7', 'FC3', 'FCZ', 'FC4', 'FT8', 'T3', 'C3', 'CZ', 'C4', 'T4', 'TP7', 'CP3', 'CPZ', 'CP4', 'TP8', 'T5', 'P3', 'PZ', 'P4', 'T6', 'O1', 'OZ', 'O2']


Set random seed as: 42
Set step size of increase file as: 250 (3.75seconds overlap)
Set step size of normal file as: 1000 (No overlap for normal class)


In [3]:
#This part of the code is just for previewing the data split based on seed number

# Paths for input EDF folders
increase_path = 'increase_edf_no400'
normal_path = 'normal_edf_no400'

output_folders = []


# Get list of all EDF files for both classes
increase_files = [f for f in os.listdir(increase_path) if f.endswith('.edf')]
normal_files = [f for f in os.listdir(normal_path) if f.endswith('.edf')]


In [4]:
train_normal_files = ['7.edf', '11.edf', '13.edf', '14.edf', '16.edf', '21.edf', '22.edf', '24.edf', '25.edf', '26.edf', '28.edf', '29.edf', '30.edf', '31.edf', '32.edf', '34.edf', '35.edf', '36.edf', '37.edf', '38.edf', '39.edf', '40.edf', '41.edf', '42.edf', '43.edf', '45.edf', '46.edf', '48.edf', '50.edf', '51.edf', '54.edf', '55.edf', '62.edf', '64.edf', '69.edf', '70.edf', '71.edf', '72.edf', '75.edf', '76.edf', '79.edf', '80.edf', '83.edf', '98.edf', '102.edf', '105.edf', '106.edf', '107.edf', '109.edf', '110.edf']
train_increase_files = ['2.edf', '3.edf', '5.edf', '6.edf', '17.edf', '19.edf', '20.edf', '27.edf', '52.edf', '58.edf', '60.edf', '63.edf', '97.edf', '103.edf', '104.edf']
val_increase_files = ['1.edf', '4.edf']
val_normal_files = ['15.edf', '23.edf', '49.edf', '53.edf', '77.edf', '108.edf']
test_increase_files = ['18.edf', '101.edf']
test_normal_files = ['8.edf', '9.edf', '10.edf', '12.edf', '33.edf', '44.edf', '84.edf']


train_increase_nums = sorted([int(f.split('.')[0]) for f in train_increase_files])
print(f"Train files(increase) [{len(train_increase_nums)}]: {train_increase_nums}")

train_normal_nums = sorted([int(f.split('.')[0]) for f in train_normal_files])
print(f"Train files(normal) [{len(train_normal_nums)}]: {train_normal_nums}")

val_increase_nums = sorted([int(f.split('.')[0]) for f in val_increase_files])
print(f"Val files(increase) [{len(val_increase_nums)}]: {val_increase_nums}")

val_normal_nums = sorted([int(f.split('.')[0]) for f in val_normal_files])
print(f"Val files(normal) [{len(val_normal_nums)}]: {val_normal_nums}")

test_increase_nums = sorted([int(f.split('.')[0]) for f in test_increase_files])
print(f"Test files(increase) [{len(test_increase_nums)}]: {test_increase_nums}")

test_normal_nums = sorted([int(f.split('.')[0]) for f in test_normal_files])
print(f"Test files(normal) [{len(test_normal_nums)}]: {test_normal_nums}")

Train files(increase) [15]: [2, 3, 5, 6, 17, 19, 20, 27, 52, 58, 60, 63, 97, 103, 104]
Train files(normal) [50]: [7, 11, 13, 14, 16, 21, 22, 24, 25, 26, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 48, 50, 51, 54, 55, 62, 64, 69, 70, 71, 72, 75, 76, 79, 80, 83, 98, 102, 105, 106, 107, 109, 110]
Val files(increase) [2]: [1, 4]
Val files(normal) [6]: [15, 23, 49, 53, 77, 108]
Test files(increase) [2]: [18, 101]
Test files(normal) [7]: [8, 9, 10, 12, 33, 44, 84]


In [5]:
# Model-specific preprocessing functions for Stress dataset
# These functions apply model-specific preprocessing to raw data before chunking
# Copied from sleep preprocessing code to ensure consistency

def df_to_raw_full(df, sfreq=200):
    ch_names = [c for c in df.columns if c not in ['Time', 'FeedBackEvent', 'EOG']]
    eeg_data = df[ch_names].T.values  # (n_ch, n_times)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data, info, verbose=False)
    return raw, ch_names


def filter_full(raw, l_freq, h_freq, line_noise=50, target_sfreq=200):
    sf = raw.info['sfreq']
    raw.filter(l_freq, h_freq, n_jobs=-1)
    raw.notch_filter(line_noise, filter_length='auto', n_jobs=-1)
    if target_sfreq and target_sfreq != sf:
        raw.resample(target_sfreq, npad='auto', verbose=False)
    return raw

def remove_dc_offset(raw):
    """Remove DC offset per channel."""
    data, times = raw.get_data(return_times=True)
    data -= np.mean(data, axis=1, keepdims=True)
    raw._data = data
    return raw

def remove_linear_trend(raw):
    """Remove linear trend per channel."""
    try:
        data = detrend(raw.get_data(), axis=1, type='linear')
        # print("✅ detrend finished successfully")
    except Exception as e:
        print("❌ detrend failed:", e)
    raw._data = data
    return raw


def normalize_amplitude(raw, mode="div100"):
    """Amplitude normalization according to mode."""
    data = raw.get_data()
    if mode == "div100":
        raw._data = data / 100.0  # µV
    elif mode == "zscore_per_channel":
        mean = np.mean(data, axis=1, keepdims=True)
        std = np.std(data, axis=1, keepdims=True)
        raw._data = (data - mean) / std
    elif mode == "global_zscore":
        raw._data = (data - np.mean(data)) / np.std(data)
    elif mode == "percentile_95":
        p95 = np.percentile(np.abs(data), 95, axis=1, keepdims=True)
        raw._data = data / p95
    return raw
    
def preprocess_stress_cbramod(raw, line_noise=50):
    """CBraMod preprocessing for Stress dataset"""
    raw.resample(200)
    raw = filter_full(raw, 0.3, 75.0, line_noise=line_noise, target_sfreq=200)
    return raw

def preprocess_stress_labram(raw, line_noise=50):
    """LaBraM preprocessing for Stress dataset"""
    raw.resample(200)
    raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    return raw

def preprocess_stress_neurolm(raw, line_noise=50):
    """NeuroLM preprocessing for Stress dataset"""
    raw.resample(200, npad="auto", verbose="error")
    raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    return raw

def preprocess_stress_biot(raw, line_noise=50):
    """BIOT preprocessing for Stress dataset"""
    raw.resample(200, npad="auto", verbose="error")
    # No filtering for BIOT
    return raw

def preprocess_stress_eegpt(raw, line_noise=50):
    """EEGPT preprocessing for Stress dataset"""
    raw.resample(256, npad="auto", verbose="error")
    raw.filter(l_freq=0.1, h_freq=75.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    raw = remove_dc_offset(raw)
    raw.set_eeg_reference(ref_channels='average', projection=False)
    return raw

def preprocess_stress_neurogpt(raw, line_noise=50):
    """NeuroGPT preprocessing for Stress dataset"""
    raw.resample(250, npad="auto", verbose="error")
    raw.filter(l_freq=0.5, h_freq=100.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    raw = remove_dc_offset(raw)
    raw = remove_linear_trend(raw)
    raw.set_eeg_reference(ref_channels="average")
    return raw

def preprocess_stress_sttransformer(raw, line_noise=50):
    """ST-Transformer preprocessing for Stress dataset"""
    raw.resample(250, npad="auto", verbose="error")
    raw.filter(l_freq=4.0, h_freq=40.0, n_jobs=1)
    raw.notch_filter(line_noise, n_jobs=1)
    return raw

# Model extractor dictionary
STRESS_PREPROCESSORS = {
    "cbramod": preprocess_stress_cbramod,
    "labram": preprocess_stress_labram,
    "neurolm": preprocess_stress_neurolm,
    "biot": preprocess_stress_biot,
    "eegpt": preprocess_stress_eegpt,
    "neurogpt": preprocess_stress_neurogpt,
    "sttransformer": preprocess_stress_sttransformer,
}

In [6]:
# Function to process EDF files: downsample, chunk, and label
def process_edf(file_path, label, sfreq=200, chunk_size=5, overlap=1, step_size=None, selected_channels=None, preprocess_func=None, line_noise=50):
    print(f"Processing: {file_path}")

    try:
        raw = mne.io.read_raw_edf(file_path, preload=True)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return []
    
    # Select only specified channels
    if selected_channels:
        raw.pick_channels(selected_channels)

    # Convert to microvolts for consistency
    raw._data *= 1e6

    # Apply model-specific preprocessing if provided
    if preprocess_func is not None:
        raw = preprocess_func(raw, line_noise=line_noise)
        # Get the actual sampling frequency after preprocessing
        sfreq = raw.info['sfreq']
    else:
        # Default: just resample if necessary
        if raw.info['sfreq'] != sfreq:
            raw.resample(sfreq)

    data = raw.get_data()  # (n_channels, n_samples)
    n_channels, n_samples = data.shape

    chunk_samples = int(chunk_size * sfreq)  # Number of samples per 5s chunk
    overlap_samples = int(overlap * sfreq)   # Number of overlapping samples

    # Use specified step size, or calculate it based on overlap
    if step_size is None:
        step_size = chunk_samples - overlap_samples

    chunks = []

    # Generate overlapping chunks
    for start in range(0, n_samples - chunk_samples + 1, step_size):
        chunk_data = data[:, start:start + chunk_samples]
        chunks.append((chunk_data, label))  # Store chunk and label as a tuple

    # Store diagnostic information
    file_name = os.path.basename(file_path)
    duration_sec = n_samples / sfreq
    diagnostic_info = {
        'file_name': file_name,
        'chunks': len(chunks),
        'sfreq': sfreq,
        'n_samples': n_samples,
        'duration_sec': duration_sec,
        'step_size': step_size,
        'label': label
    }
    
    # Print diagnostic information
    print(f"  -> {file_name}: {len(chunks)} chunks, {sfreq:.1f} Hz, {n_samples} samples ({duration_sec:.2f} sec), step_size={step_size}")
    
    # Store to global diagnostics list
    if 'file_diagnostics' not in globals():
        globals()['file_diagnostics'] = []
    globals()['file_diagnostics'].append(diagnostic_info)

    return chunks

# Helper function to process a list of files
def process_files(file_list, label, step_size, selected_channels=None, preprocess_func=None, line_noise=50):
    chunks = []
    total_chunks = 0
    for filename in file_list:
        file_path = os.path.join(increase_path if label == 1 else normal_path, filename)
        file_chunks = process_edf(file_path, label, step_size=step_size, selected_channels=selected_channels, preprocess_func=preprocess_func, line_noise=line_noise)
        chunks.extend(file_chunks)
        total_chunks += len(file_chunks)
    print(f"  Total chunks from {len(file_list)} files: {total_chunks}")
    
    # Store summary to global diagnostics
    if 'file_diagnostics' not in globals():
        globals()['file_diagnostics'] = []
    globals()['file_diagnostics'].append({
        'summary': True,
        'file_list': file_list,
        'label': label,
        'total_chunks': total_chunks,
        'num_files': len(file_list)
    })
    
    return chunks

In [7]:
# Main processing function for Stress dataset with model-specific preprocessing

def save_chunks(chunks, folder):
    for i, (chunk_data, label) in enumerate(chunks):
        filename = os.path.join(folder, f"chunk_{i}.pickle")
        with open(filename, 'wb') as f:
            pickle.dump({'X': chunk_data, 'y': label}, f)
        print(f"Saved: {filename}")
        
def process_stress_dataset(model_name="labram", channel_size=16, rs=2, 
                          increase_step_size=250, normal_step_size=1000,
                          train_increase_files=None, val_increase_files=None, test_increase_files=None,
                          train_normal_files=None, val_normal_files=None, test_normal_files=None,
                          subset_channels=None, increase_path='increase_edf_no400', normal_path='normal_edf_no400',
                          line_noise=50):
    """
    Process Stress dataset with model-specific preprocessing.
    
    Args:
        model_name: Name of the model (cbramod, labram, neurolm, biot, eegpt, neurogpt, sttransformer)
        channel_size: Number of channels
        rs: Random seed
        increase_step_size: Step size for increase class files
        normal_step_size: Step size for normal class files
        train_increase_files, val_increase_files, test_increase_files: File lists for increase class
        train_normal_files, val_normal_files, test_normal_files: File lists for normal class
        subset_channels: List of channels to use
        increase_path, normal_path: Paths to data folders
        line_noise: Line noise frequency (default 50 Hz)
    """
    # Get model-specific preprocessor
    if model_name not in STRESS_PREPROCESSORS:
        raise ValueError(f"Unknown model name: {model_name}. Available models: {list(STRESS_PREPROCESSORS.keys())}")
    
    preprocess_func = STRESS_PREPROCESSORS[model_name]
    
    # Set output path with model name prefix
    output_path = f'augmented_data/{model_name}_Stress_noleak_{channel_size}chan_no400up_swien42'
    
    # Create output folders (same format as original)
    output_folders = []
    for folder in ['train', 'val', 'test']:
        os.makedirs(f"{output_path}/{folder}", exist_ok=True)
        output_folders.append(f"{output_path}/{folder}")
        print(f"mkdir: {output_folders[-1]}")
    
    print(f"\n{'='*60}")
    print(f"Processing Stress dataset for model: {model_name}")
    print(f"Output path: {output_path}")
    print(f"{'='*60}\n")
    
    # Process files with model-specific preprocessing
    print("Processing increase files... (select channels + model-specific preprocessing + chunks)")
    train_increase = process_files(train_increase_files, label=1, step_size=increase_step_size, 
                                   selected_channels=subset_channels, preprocess_func=preprocess_func, line_noise=line_noise)
    val_increase = process_files(val_increase_files, label=1, step_size=increase_step_size, 
                                  selected_channels=subset_channels, preprocess_func=preprocess_func, line_noise=line_noise)
    test_increase = process_files(test_increase_files, label=1, step_size=increase_step_size, 
                                  selected_channels=subset_channels, preprocess_func=preprocess_func, line_noise=line_noise)
    
    print("Processing normal files... (select channels + model-specific preprocessing + chunks)")
    train_normal = process_files(train_normal_files, label=0, step_size=normal_step_size, 
                                 selected_channels=subset_channels, preprocess_func=preprocess_func, line_noise=line_noise)
    val_normal = process_files(val_normal_files, label=0, step_size=normal_step_size, 
                               selected_channels=subset_channels, preprocess_func=preprocess_func, line_noise=line_noise)
    test_normal = process_files(test_normal_files, label=0, step_size=normal_step_size, 
                                selected_channels=subset_channels, preprocess_func=preprocess_func, line_noise=line_noise)
    
    # Combine chunks
    train_chunks = train_increase + train_normal
    val_chunks = val_increase + val_normal
    test_chunks = test_increase + test_normal
    
    # Shuffle data
    print("Shuffling data...")
    random.seed(rs)
    random.shuffle(train_chunks)
    random.shuffle(val_chunks)
    random.shuffle(test_chunks)
    
    # Save data using original save_chunks function
    print("Saving data...")
    save_chunks(train_chunks, output_folders[0])
    save_chunks(val_chunks, output_folders[1])
    save_chunks(test_chunks, output_folders[2])
    
    # Save diagnostic information
    if 'file_diagnostics' in globals() and len(globals()['file_diagnostics']) > 0:
        import json
        import numpy as np
        
        diagnostics = globals()['file_diagnostics']
        diag_file_json = os.path.join(output_path, 'file_diagnostics.json')
        diag_file_npy = os.path.join(output_path, 'file_diagnostics.npy')
        
        # Save as JSON (human-readable)
        with open(diag_file_json, 'w', encoding='utf-8') as f:
            json.dump(diagnostics, f, indent=2, ensure_ascii=False)
        print(f"Saved diagnostics to: {diag_file_json}")
        
        # Save as NPY (for easy loading)
        np.save(diag_file_npy, diagnostics, allow_pickle=True)
        print(f"Saved diagnostics to: {diag_file_npy}")
        
        # Also save as a simple variable for easy access
        globals()['diagnostics'] = diagnostics
        print(f"Diagnostics also available as variable 'diagnostics'")
    
    print(f"\n{'='*60}")
    print(f"Completed processing for model: {model_name}")
    print(f"Train chunks: {len(train_chunks)}, Val chunks: {len(val_chunks)}, Test chunks: {len(test_chunks)}")
    print(f"{'='*60}\n")
    
    return output_path

In [11]:


# for model_name in ["cbramod", "labram", "neurolm", "biot", "eegpt", "neurogpt", "sttransformer"]:
#     process_stress_dataset(
#         model_name=model_name,
#         channel_size=channel_size,
#         rs=rs,
#         increase_step_size=increase_step_size,
#         normal_step_size=normal_step_size,
#         train_increase_files=train_increase_files,
#         val_increase_files=val_increase_files,
#         test_increase_files=test_increase_files,
#         train_normal_files=train_normal_files,
#         val_normal_files=val_normal_files,
#         test_normal_files=test_normal_files,
#         subset_channels=subset_channels,
#         increase_path=increase_path,
#         normal_path=normal_path
#     )

In [ ]:
model_name = "eegpt"
process_stress_dataset(
        model_name=model_name,
        channel_size=channel_size,
        rs=rs,
        increase_step_size=increase_step_size,
        normal_step_size=normal_step_size,
        train_increase_files=train_increase_files,
        val_increase_files=val_increase_files,
        test_increase_files=test_increase_files,
        train_normal_files=train_normal_files,
        val_normal_files=val_normal_files,
        test_normal_files=test_normal_files,
        subset_channels=subset_channels,
        increase_path=increase_path,
        normal_path=normal_path
    )

In [9]:
training_directory = './augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val' #output_folders[2]  # Update this path to your training directory


def inspect_label_ratios(directory):
    total_labels = 0
    class_labels = {0: 0, 1: 0}  # Assuming binary labels: 0 for normal and 1 for increase

    for filename in os.listdir(directory):
        if filename.endswith('.pickle'):
            file_path = os.path.join(directory, filename)
            print(f"Processing {filename} from {directory}...")

            try:
                # Load the pickle file
                with open(file_path, 'rb') as f:
                    data_dict = pickle.load(f)

                # Check if the expected keys are in the dictionary
                if 'y' not in data_dict:
                    print(f"Warning: 'y' key not found in {filename}. Skipping this file.")
                    continue
                
                labels = data_dict['y']

                # Check the type of labels
                if isinstance(labels, list):
                    labels = np.array(labels)  # Convert list to numpy array
                elif isinstance(labels, int):
                    labels = np.array([labels])  # Convert single integer to a numpy array
                elif not isinstance(labels, np.ndarray):
                    print(f"Warning: Unexpected label format in {filename}. Skipping this file.")
                    continue

                # Count the labels
                total_labels += len(labels)
                class_labels[1] += np.sum(labels)  # Count positive class (increase)
                class_labels[0] += len(labels) - np.sum(labels)  # Count negative class (normal)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Calculate the ratios
    if total_labels > 0:
        increase_ratio = class_labels[1] / total_labels
        normal_ratio = class_labels[0] / total_labels
    else:
        increase_ratio = 0
        normal_ratio = 0

    print(f"\nLabel Ratios:")
    print(f"Increase Class Ratio: {increase_ratio:.4f} (total: {total_labels}, class count: {class_labels[1]})")
    print(f"Normal Class Ratio: {normal_ratio:.4f} (total: {total_labels}, class count: {class_labels[0]})")

# Inspect the label ratios for all classes in the training directory
inspect_label_ratios(training_directory)

Processing chunk_24.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_822.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_580.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_737.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_59.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_480.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_637.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_166.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_754.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing chunk_654.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/val...
Processing c

In [10]:
training_directory = './augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test' #output_folders[2]  # Update this path to your training directory


def inspect_label_ratios(directory):
    total_labels = 0
    class_labels = {0: 0, 1: 0}  # Assuming binary labels: 0 for normal and 1 for increase

    for filename in os.listdir(directory):
        if filename.endswith('.pickle'):
            file_path = os.path.join(directory, filename)
            print(f"Processing {filename} from {directory}...")

            try:
                # Load the pickle file
                with open(file_path, 'rb') as f:
                    data_dict = pickle.load(f)

                # Check if the expected keys are in the dictionary
                if 'y' not in data_dict:
                    print(f"Warning: 'y' key not found in {filename}. Skipping this file.")
                    continue
                
                labels = data_dict['y']

                # Check the type of labels
                if isinstance(labels, list):
                    labels = np.array(labels)  # Convert list to numpy array
                elif isinstance(labels, int):
                    labels = np.array([labels])  # Convert single integer to a numpy array
                elif not isinstance(labels, np.ndarray):
                    print(f"Warning: Unexpected label format in {filename}. Skipping this file.")
                    continue

                # Count the labels
                total_labels += len(labels)
                class_labels[1] += np.sum(labels)  # Count positive class (increase)
                class_labels[0] += len(labels) - np.sum(labels)  # Count negative class (normal)

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Calculate the ratios
    if total_labels > 0:
        increase_ratio = class_labels[1] / total_labels
        normal_ratio = class_labels[0] / total_labels
    else:
        increase_ratio = 0
        normal_ratio = 0

    print(f"\nLabel Ratios:")
    print(f"Increase Class Ratio: {increase_ratio:.4f} (total: {total_labels}, class count: {class_labels[1]})")
    print(f"Normal Class Ratio: {normal_ratio:.4f} (total: {total_labels}, class count: {class_labels[0]})")

# Inspect the label ratios for all classes in the training directory
inspect_label_ratios(training_directory)

Processing chunk_24.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_822.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_580.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_737.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_59.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_480.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_637.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_166.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_754.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Processing chunk_654.pickle from ./augmented_data/biot_Stress_noleak_30chan_no400up_swien42/test...
Pr